# 📈 CodeAlpha Data Science Internship: Task 4
## Sales Prediction with Python (Advertising Impact Analysis)

**Intern Name**: Shreya  
**Organization**: [CodeAlpha](https://www.codealpha.tech)  
**Task**: Forecast future sales based on advertising investments across TV, Radio, and Newspaper channels.

### 1. Import Essential Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib

sns.set_theme(style='whitegrid')

### 2. Ingest and Inspect Advertising Dataset

In [ ]:
df = pd.read_csv('../data/advertising.csv')
unnamed_cols = [c for c in df.columns if 'Unnamed' in c or c == '']
if unnamed_cols:
    df = df.drop(columns=unnamed_cols)

print(f'Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

In [ ]:
print('Missing Values Check:')
print(df.isnull().sum())

df.describe()

### 3. Exploratory Data Analysis & Visualizations
Analyzing how individual media advertising channels impact sales volume.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
channels = ['TV', 'Radio', 'Newspaper']
for ax, ch in zip(axes, channels):
    sns.regplot(data=df, x=ch, y='Sales', ax=ax, line_kws={'color': 'red'})
    ax.set_title(f'{ch} vs. Sales')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(6, 4))
sns.heatmap(df.corr(), annot=True, cmap='YlGnBu', fmt='.3f')
plt.title('Correlation Matrix: Advertising vs Sales', weight='bold')
plt.show()

### 4. Train / Test Split

In [ ]:
X = df[['TV', 'Radio', 'Newspaper']]
y = df['Sales']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Training Set: {X_train.shape[0]} samples')
print(f'Test Set: {X_test.shape[0]} samples')

### 5. Multi-Model Regression Benchmarking
Comparing Linear Regression, Ridge, and Random Forest Regressor.

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=100, random_state=42)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    print(f'=== {name} ===')
    print(f'Test R² Score : {r2:.4f}')
    print(f'MAE           : {mae:.4f}')
    print(f'RMSE          : {rmse:.4f}\n')

### 6. Marketing Insights & Ad Spend Optimization

In [ ]:
lr = models['Linear Regression']
coef_df = pd.DataFrame({
    'Channel': X.columns,
    'Coefficient': lr.coef_
}).sort_values(by='Coefficient', ascending=False)

print(f'Base Sales (Zero Ad Spend): {lr.intercept_:.2f}k units\n')
print(coef_df.to_string(index=False))

plt.figure(figsize=(6, 3))
sns.barplot(data=coef_df, x='Channel', y='Coefficient', palette='Blues_r')
plt.title('Sales Boost per $1,000 Spend by Channel', weight='bold')
plt.ylabel('Incremental Sales Units (k)')
plt.show()

### 7. Strategic Forecasting Scenario

In [ ]:
new_campaign = pd.DataFrame([{'TV': 150.0, 'Radio': 25.0, 'Newspaper': 15.0}])
predicted_sales = models['Random Forest Regressor'].predict(new_campaign)[0]
print(f'Expected Sales for Campaign: {predicted_sales:.2f} thousand units')